In [4]:
# Install everything needed
!pip install transformers datasets trl accelerate

In [5]:
import os
import re
import random
import torch
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    AutoModelForCausalLM,
    AutoTokenizer
)
from trl import GRPOTrainer, GRPOConfig, PPOTrainer, PPOConfig
from tqdm import tqdm
from typing import List

import copy, torch, torch.nn as nn
from transformers import AutoTokenizer, GPT2Config, GPT2Model
from trl import (
    PPOConfig, PPOTrainer,
    AutoModelForCausalLMWithValueHead,
    create_reference_model,
)
from transformers.modeling_outputs import CausalLMOutputWithCrossAttentions

In [6]:
# --- Settings ---
model_name = "gpt2"  # GPT-2 small
output_dir_sft = "./gpt2_sft"
output_dir_grpo = "./gpt2_grpo"
output_dir_ppo = "./gpt2_ppo"

In [7]:
# --- Load dataset ---
dataset = load_dataset("openai/gsm8k", "main")["train"]
dataset = dataset.shuffle(seed=42)

# --- Take a small subset ---
sft_dataset = dataset.select(range(1000))   # 1000 samples for SFT
grpo_dataset = dataset.select(range(1000, 7000))  # 6000 samples for GRPO & PPO

# --- Tokenizer ---
tokenizer = GPT2Tokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

# --- Prepare SFT dataset ---
def format_sft(example):
    prompt = f"{example['question']}"
    target = f" {example['answer']}"
    text = prompt + target
    return tokenizer(text, truncation=True, padding="max_length", max_length=512)

sft_dataset = sft_dataset.map(format_sft)

# --- Load GPT-2-small for SFT ---
model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# --- SFT Training setup ---
training_args = TrainingArguments(
    output_dir=output_dir_sft,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="no",
    logging_steps=10,
    learning_rate=5e-5,
    bf16=True if torch.cuda.is_available() else False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sft_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting Supervised Fine-Tuning (SFT)...")
trainer.train()
print("Finished SFT!")

# Save SFT model
model.save_pretrained(output_dir_sft)
tokenizer.save_pretrained(output_dir_sft)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
<ipython-input-7-d5ab5e15a76a>:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Starting Supervised Fine-Tuning (SFT)...


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: muhammad163 (muhammad163-minerva-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,6.863000
20,5.194900
30,4.281800
40,3.950900
50,3.741000
60,3.430800
70,3.316500
80,3.150100
90,2.996300
100,2.728000


Finished SFT!


('./gpt2_sft/tokenizer_config.json',
 './gpt2_sft/special_tokens_map.json',
 './gpt2_sft/vocab.json',
 './gpt2_sft/merges.txt',
 './gpt2_sft/added_tokens.json')

In [8]:
def evaluate_expression(expr):
    # Safe evaluation (only allow basic math operators)
    return eval(expr, {"__builtins__": None})

def format_reward_func(completions: List[str], **kwargs) -> List[float]:
    rewards = []
    for completion in completions:
        score = 0.0
        if re.search(r"<<.*?=.*?>>", completion):
            score += 0.2
        if "####" in completion:
            score += 0.2
            if completion.count("####") == 1:
                score += 0.1
            parts = completion.split("####")
            if len(parts) == 2 and "<<" in parts[0]:
                score += 0.2
        eos_token = "<|im_end|>"
        if completion.strip().endswith(eos_token):
            score += 0.2
        rewards.append(score)
    return rewards

def calculation_reward_func(completions: List[str], **kwargs) -> List[float]:
    rewards = []
    for completion in completions:
        score = 0.0
        calculations = extract_calculations(completion)
        if not calculations:
            rewards.append(0.0)
            continue
        total_calcs = len(calculations)
        correct_calcs = 0
        for expr, result in calculations:
            try:
                expected = evaluate_expression(expr)
                actual = evaluate_expression(result)
                if abs(expected - actual) < 0.01:
                    correct_calcs += 1
            except:
                continue
        if total_calcs > 0:
            score = (correct_calcs / total_calcs) * 0.5
        rewards.append(score)
    return rewards

def answer_correctness_reward_func(completions: List[str], answer: List[str], **kwargs) -> List[float]:
    rewards = []
    for completion, expected in zip(completions, answer):
        score = 0.0
        extracted = extract_final_answer(completion)
        if extracted is not None and expected is not None:
            extracted = extracted.replace(" ", "").replace(",", "").replace("$", "").strip()
            expected = str(expected).replace(" ", "").replace(",", "").replace("$", "").strip()
            try:
                if abs(float(extracted) - float(expected)) < 0.01:
                    score = 1.0
            except:
                if extracted == expected:
                    score = 1.0
        rewards.append(score)
    return rewards

def combined_reward_func(prompts: List[str], completions: List[str], **kwargs) -> List[float]:
    answers = kwargs.get("answer", None)
    if answers is None:
        raise ValueError("Answers not provided to reward function!")

    format_rewards = format_reward_func(completions)
    calc_rewards = calculation_reward_func(completions)
    answer_rewards = answer_correctness_reward_func(completions, answers)

    combined_rewards = []
    for f, c, a in zip(format_rewards, calc_rewards, answer_rewards):
        reward = (f * 0.05) + (c * 0.05) + (a * 0.9)
        combined_rewards.append(reward)
    return combined_rewards

In [6]:
# --- Reload the SFT model and tokenizer ---
model = AutoModelForCausalLM.from_pretrained(output_dir_sft)
tokenizer = AutoTokenizer.from_pretrained(output_dir_sft)
model.resize_token_embeddings(len(tokenizer))

# --- Prepare GRPO dataset ---
def format_grpo(example):
    return {"prompt": f"{example['question']}"}

grpo_dataset = grpo_dataset.map(format_grpo)

# --- Reward functions (yours) ---
def extract_calculations(text):
    matches = re.findall(r"<<(.*?)=(.*?)>>", text)
    return [(expr.strip(), result.strip()) for expr, result in matches]

def extract_final_answer(text):
    if "####" in text:
        parts = text.split("####")
        if len(parts) > 1:
            after = parts[1].strip()
            if after:
                return after.split()[0]
    return None

# --- GRPO Config ---
grpo_config = GRPOConfig(
    output_dir=output_dir_grpo,
    per_device_train_batch_size=32,
    learning_rate=1e-6,
    logging_steps=100,
    num_train_epochs=1,
    max_prompt_length=512,
    max_completion_length=128,
    num_generations=4,
    temperature=0.7,
)

# --- GRPO Trainer ---
grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=combined_reward_func,
)

print("🚀 Starting GRPO Training...")
grpo_trainer.train()
print("🏁 Finished GRPO Training!")

# Save the model
grpo_trainer.save_model(output_dir_grpo)
tokenizer.save_pretrained(output_dir_grpo)

print(f"✅ GRPO model saved to {output_dir_grpo}")

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

🚀 Starting GRPO Training...


Step,Training Loss
100,217.010100
200,27.559200
300,14.185500
400,4.545800
500,7.069900
600,16.769200
700,33.742100


<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'float' object is not callable; perhaps you missed a comma?


🏁 Finished GRPO Training!
✅ GRPO model saved to ./gpt2_grpo


In [10]:
# ===================================================================================
# 🎯  PPO fine-tuning – inspired by GRPO, fully fixed for trl-0.17.0
# ===================================================================================

# ------------------------------------------------------------------------------
# 1️⃣  Load SFT checkpoint + tokenizer
# ------------------------------------------------------------------------------
model = AutoModelForCausalLMWithValueHead.from_pretrained(output_dir_sft)
tokenizer = AutoTokenizer.from_pretrained(output_dir_sft)
tokenizer.pad_token = tokenizer.eos_token
model.pretrained_model.resize_token_embeddings(len(tokenizer))

# --- Patch model.forward() to always return ModelOutput ---
old_forward = model.forward
def wrapped_forward(*args, **kwargs):
    outputs = old_forward(*args, **kwargs)
    if isinstance(outputs, tuple):
        logits = outputs[0]
        return CausalLMOutputWithCrossAttentions(logits=logits)
    return outputs
model.forward = wrapped_forward

# patch: add dummy .score method to avoid AttributeError
def dummy_score(hidden_states):
    return torch.zeros(hidden_states.size()[:-1], device=hidden_states.device)
model.score = dummy_score

# make wrapper look like vanilla HF model
if not hasattr(model, "generation_config"):
    model.generation_config = copy.deepcopy(model.pretrained_model.generation_config)
bp = model.pretrained_model.base_model_prefix
model.base_model_prefix = bp
setattr(model, bp, model.pretrained_model)

# ------------------------------------------------------------------------------
# 2️⃣  Reference model (frozen baseline for KL)
# ------------------------------------------------------------------------------
ref_model = create_reference_model(model.pretrained_model)
for m in (model.pretrained_model, ref_model):
    m.config.return_dict = True

In [11]:
# ------------------------------------------------------------------------------
# 3️⃣  Reward model that obeys PPOTrainer’s interface
# ------------------------------------------------------------------------------
class RewardModel(nn.Module):
    def __init__(self, backbone, tokenizer, reward_fn):
        super().__init__()
        self.backbone = backbone
        self.base_model_prefix = "backbone"
        self.tokenizer = tokenizer
        self.reward_fn = reward_fn
        self.v_head = nn.Sequential(
            nn.Linear(backbone.config.n_embd, 1),
            nn.Tanh()
        )

    def forward(self, input_ids, attention_mask=None, position_ids=None,
                use_cache=False, output_hidden_states=True):
        outputs = self.backbone(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            use_cache=use_cache,
            output_hidden_states=output_hidden_states,
            return_dict=True
        )
        return outputs

    def score(self, hidden_states):
        return self.v_head(hidden_states).squeeze(-1)

    def text_reward(self, queries, responses):
        return torch.tensor(
            self.reward_fn(queries, responses),
            dtype=torch.float32,
            device=self.backbone.device,
        )

reward_model = RewardModel(
    backbone   = model.pretrained_model,
    tokenizer  = tokenizer,
    reward_fn  = combined_reward_func,
)

In [12]:
# ------------------------------------------------------------------------------
# 4️⃣  Dataset build (train & eval)
# ------------------------------------------------------------------------------
def to_ppo(sample):
    return {"query": f"{sample['question']}"}

ppo_ds = grpo_dataset.map(to_ppo, remove_columns=grpo_dataset.column_names)

def collate(batch):
    return tokenizer(
        [b["query"] for b in batch],
        padding=True, truncation=True, return_tensors="pt",
    )

# Use same data for evaluation (small subset)
ppo_eval_ds = ppo_ds.select(range(min(100, len(ppo_ds))))  # 100 examples max

# ------------------------------------------------------------------------------
# 5️⃣  PPO hyper-parameters
# ------------------------------------------------------------------------------
ppo_cfg = PPOConfig(
    batch_size=16,
    mini_batch_size=32,
    num_ppo_epochs=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-6,
    seed=42,
    temperature=0.7,
    output_dir=output_dir_grpo,
    save_safetensors = False,
)

# ------------------------------------------------------------------------------
# 6️⃣  Initialise PPOTrainer
# ------------------------------------------------------------------------------
trainer = PPOTrainer(
    ppo_cfg,
    tokenizer,
    model,
    ref_model,
    reward_model,
    ppo_ds,
    value_model=model,
    data_collator=collate,
    eval_dataset=ppo_eval_ds,
)

# ------------------------------------------------------------------------------
# 7️⃣  Train
# ------------------------------------------------------------------------------
print("🚀  Starting PPO training …")
trainer.train()
print("🏁  Training complete!")

# ------------------------------------------------------------------------------
# 8️⃣  Save everything
# ------------------------------------------------------------------------------
trainer.save_model(output_dir_ppo)
tokenizer.save_pretrained(output_dir_ppo)
print(f"✅  Saved PPO-tuned model to {output_dir_grpo}")

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 50256, 'eos_token_id': 50256}. If this is not desired, please set these values explicitly.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


🚀  Starting PPO training …
===training policy===


/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:638: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  metrics["val/ratio_var"] = self.accelerator.gather_for_metrics(ratio_stats).var().item()


Step,Training Loss


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ query                                       ┃ model response                              ┃ score               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ Sally sold 20 cups of lemonade last week.   │  Sally sold 20 cups of lemonade last week.  │ -0.9997153878211975 │
│ She sold 30% more lemonade this week. How   │ She sold 30% more lemonade this week than   │                     │
│ many cups of lemonade did she sell in total │ Sally sold 20 cups of lemonade last week.   │                     │
│ for both weeks?                             │ She sold 30% more lemonade this week than   │                     │
│                                             │ Sally sold 20 cups of lemonade last week.   │                     │
│                                             │                                             │                     │
├─────────────────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────┤
│ A parking area near Peter's house is 4      │  The total number of parking spots in all   │ -0.9790014028549194 │
│ stories tall. There are 4 open parking      │ is 4*4=<<4*4=14>>14                         │                     │
│ spots on the first level. There are 7 more  │ The total number of parking spots in all is │                     │
│ open parking spots on the second level than │ 14*14=<<14*14=16>>16                        │                     │
│ on the first level, and there are 6 more    │ The total number of parking spots in all is │                     │
│ open parking spots on the third level than  │                                             │                     │
│ on the second level. There are 14 open      │                                             │                     │
│ parking spots on the fourth level. How many │                                             │                     │
│ open parking spots are there in all?        │                                             │                     │
├─────────────────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────┤
│ Vincent's bookstore is divided into         │  He will earn $4 + $4 = $<<4+4=12>>12 after │ 0.09116165339946747 │
│ different kinds of books. His top-selling   │ five days.                                  │                     │
│ books are fantasy books. He also sells      │ He will earn $12 + $12 = $<<12+12=24>>24    │                     │
│ literature books which cost half of the     │ after 24 days.                              │                     │
│ price of a fantasy book. If his fantasy     │ He will earn $24 + $                        │                     │
│ books cost $4 each, and he sold five        │                                             │                     │
│ fantasy books and eight literature books    │                                             │                     │
│ per day, how much money will he earn after  │                                             │                     │
│ five days?                                  │                                             │                     │
├─────────────────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────┤
│ Carla needs to bring water to her animals.  │  Carla needs to bring 8 gallons of water to │ -0.9999486207962036 │
│ Each horse needs twice as much water as a   │ her animals. Each horse needs twice as much │                     │
│ pig, and the chickens drink from one tank   │ water as a pig, and the chickens drink from │                     │
│ that needs 30 gallons. How many gallons of  │ one tank that needs 30 gallons.             │                     │
│ water does Carla need to bring if she has 8 │ Carla ne

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:638: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  metrics["val/ratio_var"] = self.accelerator.gather_for_metrics(ratio_stats).var().item()
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ query                                       ┃ model response                              ┃ score               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ Sally sold 20 cups of lemonade last week.   │  Sally sold 20 cups of lemonade last week.  │ -0.999707043170929  │
│ She sold 30% more lemonade this week. How   │ She sold 30% more lemonade this week than   │                     │
│ many cups of lemonade did she sell in total │ Sally sold 20 cups of lemonade last week.   │                     │
│ for both weeks?                             │ She sold 30% more lemonade this week than   │                     │
│                                             │ Sally sold 20 cups of lemonade last week.   │                     │
│                                             │                                             │                     │
├─────────────────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────┤
│ A parking area near Peter's house is 4      │  The total number of parking spots in all   │ -0.9784748554229736 │
│ stories tall. There are 4 open parking      │ is 4*4=<<4*4=14>>14                         │                     │
│ spots on the first level. There are 7 more  │ The total number of parking spots in all is │                     │
│ open parking spots on the second level than │ 14*14=<<14*14=16>>16                        │                     │
│ on the first level, and there are 6 more    │ The total number of parking spots in all is │                     │
│ open parking spots on the third level than  │                                             │                     │
│ on the second level. There are 14 open      │                                             │                     │
│ parking spots on the fourth level. How many │                                             │                     │
│ open parking spots are there in all?        │                                             │                     │
├─────────────────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────┤
│ Vincent's bookstore is divided into         │  He will earn $4 + $4 = $<<4+4=12>>12 after │ 0.10767851769924164 │
│ different kinds of books. His top-selling   │ five days.                                  │                     │
│ books are fantasy books. He also sells      │ He will earn $12 + $12 = $<<12+12=24>>24    │                     │
│ literature books which cost half of the     │ after 24 days.                              │                     │
│ price of a fantasy book. If his fantasy     │ He will earn $24 + $                        │                     │
│ books cost $4 each, and he sold five        │                                             │                     │
│ fantasy books and eight literature books    │                                             │                     │
│ per day, how much money will he earn after  │                                             │                     │
│ five days?                                  │                                             │                     │
├─────────────────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────┤
│ Carla needs to bring water to her animals.  │  Carla needs to bring 8 gallons of water to │ -0.9999465346336365 │
│ Each horse needs twice as much water as a   │ her animals. Each horse needs twice as much │                     │
│ pig, and the chickens drink from one tank   │ water as a pig, and the chickens drink from │                     │
│ that needs 30 gallons. How many gallons of  │ one tank that needs 30 gallons.             │                     │
│ water does Carla need to bring if she has 8 │ Carla ne

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:638: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  metrics["val/ratio_var"] = self.accelerator.gather_for_metrics(ratio_stats).var().item()
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_

🏁  Training complete!
✅  Saved PPO-tuned model to ./gpt2_grpo


In [2]:
import re, pathlib, pandas as pd, torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [13]:
def _extract_numeric(text: str):
    """ gsm8k – pull the number after the final `####` """
    m = re.search(r"####\s*([^\s<]+)", text)
    return m.group(1).strip() if m else None


def _extract_math(text: str):
    """
    MATH-500 – grab whatever the model put in \\boxed{…};
    fall back to the last non-empty line.
    """
    m = re.search(r"\\boxed\{([^}]*)\}", text)
    if m:
        return m.group(1).strip()
    # last line, strip $\,$ and whitespace
    return re.sub(r"\s+|\$", "", text.strip().split("\n")[-1])


def evaluate_and_save(
    model_path: str,
    dataset: str = "gsm8k",            # "gsm8k" | "math500"
    batch_size: int = 32,
    max_new_tokens: int = 128,
    device: str | None = None,
    out_dir: str = "results",
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # ── model & tokenizer ───────────────────────────────────────────────────
    model     = AutoModelForCausalLM.from_pretrained(model_path).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token

    # ── dataset-specific setup ──────────────────────────────────────────────
    if dataset.lower() == "gsm8k":
        ds        = load_dataset("openai/gsm8k", "main", split="test")
        prompts   = [f"<|im_start|> {ex['question']} <|im_sep|>" for ex in ds]
        answers   = [ex["answer"] for ex in ds]
        extractor = _extract_numeric
        normalise = lambda s: re.sub(r"[,$\s]", "", s or "")
    elif dataset.lower() in {"math500", "math-500"}:
        ds        = load_dataset("HuggingFaceH4/MATH-500", split="test")
        prompts   = [f"<|im_start|> {ex['problem']} <|im_sep|>" for ex in ds]
        answers   = [ex["answer"] for ex in ds]
        extractor = _extract_math
        normalise = lambda s: re.sub(r"\s+|\$", "", s or "")
    else:
        raise ValueError(f"Unknown dataset {dataset}")

    # ── generate ────────────────────────────────────────────────────────────
    model.eval()
    preds, extracted = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(prompts), batch_size), desc=f"{model_path}:{dataset}"):
            batch  = prompts[i : i + batch_size]
            inputs = tokenizer(batch,
                               return_tensors="pt",
                               padding=True,
                               truncation=True,
                               max_length=512).to(device)

            outputs = model.generate(**inputs,
                                     max_new_tokens=max_new_tokens,
                                     do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=False))

    # ── score ───────────────────────────────────────────────────────────────
    correct = 0
    for p, gt in zip(preds, answers):
        pa = extractor(p)
        extracted.append(pa)
        if pa is None:
            continue
        correct += normalise(pa) == normalise(gt)

    accuracy = 100 * correct / len(answers)

    # ── save CSV ────────────────────────────────────────────────────────────
    pathlib.Path(out_dir).mkdir(exist_ok=True)
    df = pd.DataFrame({
        "prompt"             : prompts,
        "ground_truth"       : answers,
        "prediction_full"    : preds,
        "prediction_extracted": extracted,
    })
    csv_path = pathlib.Path(out_dir) / f"{pathlib.Path(model_path).name}_{dataset}.csv"
    df.to_csv(csv_path, index=False)

    return accuracy, correct, len(answers), csv_path

In [14]:
# ── run ────────────────────────────────────────────────────────────────────────
for ds_name in ("gsm8k", "math500"):
    for ckpt in ["./gpt2_grpo", "./gpt2_ppo", "./gpt2_sft"]:
        acc, correct, total, csv_path = evaluate_and_save(ckpt, dataset=ds_name)
        print(f"{pathlib.Path(ckpt).name:>10} | {ds_name:7}: {acc:6.2f}% ({correct}/{total}) → {csv_path}")

./gpt2_grpo:gsm8k: 100%|██████████| 42/42 [01:05<00:00,  1.56s/it]


 gpt2_grpo | gsm8k  :   0.00% (0/1319) → results/gpt2_grpo_gsm8k.csv


./gpt2_ppo:gsm8k: 100%|██████████| 42/42 [01:04<00:00,  1.55s/it]


  gpt2_ppo | gsm8k  :   0.00% (0/1319) → results/gpt2_ppo_gsm8k.csv


./gpt2_sft:gsm8k: 100%|██████████| 42/42 [01:04<00:00,  1.54s/it]


...

 gpt2_grpo | math500:   0.00% (0/500) → results/gpt2_grpo_math500.csv


./gpt2_ppo:math500: 100%|██████████| 16/16 [00:26<00:00,  1.67s/it]


  gpt2_ppo | math500:   0.00% (0/500) → results/gpt2_ppo_math500.csv


./gpt2_sft:math500: 100%|██████████| 16/16 [00:26<00:00,  1.65s/it]

  gpt2_sft | math500:   0.00% (0/500) → results/gpt2_sft_math500.csv
